In [17]:
import sys
!{sys.executable} -m pip install torch

In [18]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# Kiểm tra GPU (CUDA) nếu có, ngược lại dùng CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Đang sử dụng thiết bị: {device}")

✅ Đang sử dụng thiết bị: cpu


In [19]:
DATA_PROCESSED_PATH = "../data/processed/patient_sequences.json"

with open(DATA_PROCESSED_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

sequences = data["sequences"]
splits = data["splits"]

# 1. Thu thập toàn bộ tập từ vựng mã ICD-9
unique_codes = set()
for patient_id, visits in sequences.items():
    for visit in visits:
        for code in visit:
            unique_codes.add(code)

code2idx = {code: idx + 1 for idx, code in enumerate(sorted(list(unique_codes)))}
idx2code = {idx: code for code, idx in code2idx.items()}
vocab_size = len(code2idx) + 1  # 0 dành cho Padding

print(f"Tổng số mã ICD-9 độc lập trong tập dữ liệu: {len(code2idx)}")

# 2. Định nghĩa PyTorch Custom Dataset
class CustomEHRDataset(Dataset):
    def __init__(self, patient_ids, sequences, code2idx, vocab_size):
        self.samples = []
        for pid in patient_ids:
            visits = sequences[pid]
            if len(visits) < 2:
                continue
            
            # Chuyển đổi mã ICD-9 sang danh sách các chỉ số (indices)
            seq_indices = [[code2idx[c] for c in visit if c in code2idx] for visit in visits]
            
            # Input: từ lượt 0 -> lượt (N-1)
            # Target: từ lượt 1 -> lượt N (dự đoán lượt khám tiếp theo)
            input_seq = seq_indices[:-1]
            target_seq = seq_indices[1:]
            
            self.samples.append((input_seq, target_seq))
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        return self.samples[idx]

# 3. Hàm Collate để biến đổi danh sách lượt khám thành Multi-Hot Matrix
def collate_fn(batch):
    # Lấy phần tử đầu tiên của batch (vì batch_size = 1)
    input_seq, target_seq = batch[0]
    
    num_visits = len(input_seq)
    inputs_matrix = torch.zeros((num_visits, vocab_size), dtype=torch.float32)
    targets_matrix = torch.zeros((num_visits, vocab_size), dtype=torch.float32)
    
    for t, visit in enumerate(input_seq):
        for code_idx in visit:
            inputs_matrix[t, code_idx] = 1.0
            
    for t, visit in enumerate(target_seq):
        for code_idx in visit:
            targets_matrix[t, code_idx] = 1.0
            
    return inputs_matrix.unsqueeze(0), targets_matrix.unsqueeze(0) # [Batch=1, Seq_len, Vocab_size]

# Khởi tạo DataLoader cho tập Train, Val, Test
train_dataset = CustomEHRDataset(splits["train"], sequences, code2idx, vocab_size)
val_dataset = CustomEHRDataset(splits["val"], sequences, code2idx, vocab_size)
test_dataset = CustomEHRDataset(splits["test"], sequences, code2idx, vocab_size)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

print(f"Số lượng mẫu huấn luyện (Train): {len(train_dataset)}")
print(f"Số lượng mẫu kiểm thử (Test): {len(test_dataset)}")

Tổng số mã ICD-9 độc lập trong tập dữ liệu: 570
Số lượng mẫu huấn luyện (Train): 18
Số lượng mẫu kiểm thử (Test): 6


In [20]:
class DoctorAI(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout_prob=0.2):
        super(DoctorAI, self).__init__()
        
        # Tầng nhúng tuyến tính từ Multi-hot sang không gian Dense
        self.embedding = nn.Linear(vocab_size, embed_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_prob)
        
        # Tầng mạng nơ-ron hồi quan GRU
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        
        # Tầng phân loại đa nhãn đầu ra
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x):
        # x: [Batch_size, Seq_len, Vocab_size]
        emb = self.dropout(self.relu(self.embedding(x)))
        gru_out, _ = self.gru(emb) # gru_out: [Batch_size, Seq_len, Hidden_dim]
        logits = self.fc_out(gru_out) # logits: [Batch_size, Seq_len, Vocab_size]
        return logits

model = DoctorAI(vocab_size=vocab_size, embed_dim=128, hidden_dim=128).to(device)
print(model)

DoctorAI(
  (embedding): Linear(in_features=571, out_features=128, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
  (gru): GRU(128, 128, batch_first=True)
  (fc_out): Linear(in_features=128, out_features=571, bias=True)
)


In [21]:
def evaluate_recall_at_k(model, data_loader, k_list=[10, 20, 30]):
    model.eval()
    recalls = {k: [] for k in k_list}
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            logits = model(inputs) # [1, Seq_len, Vocab_size]
            probs = torch.sigmoid(logits).squeeze(0) # [Seq_len, Vocab_size]
            targets = targets.squeeze(0)
            
            for t in range(targets.size(0)):
                # Dùng .nonzero() để lấy vị trí các nhãn thực tế đúng chuẩn PyTorch Tensor
                true_indices = (targets[t] == 1.0).nonzero(as_tuple=False).squeeze(-1).cpu().numpy()
                if true_indices.ndim == 0:
                    true_indices = np.array([true_indices.item()])
                if len(true_indices) == 0:
                    continue
                    
                # Lấy top k dự đoán có xác suất cao nhất
                for k in k_list:
                    topk_indices = torch.topk(probs[t], k=min(k, probs.size(0))).indices.cpu().numpy()
                    hits = len(set(true_indices).intersection(set(topk_indices)))
                    recall_val = hits / len(true_indices)
                    recalls[k].append(recall_val)
                    
    avg_recalls = {k: np.mean(recalls[k]) if len(recalls[k]) > 0 else 0.0 for k in k_list}
    return avg_recalls

In [22]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20
best_val_recall30 = 0.0

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)
best_model_path = os.path.join(MODEL_DIR, "doctor_ai_best.pt")

print(" Bắt đầu quá trình huấn luyện mô hình Doctor AI...")

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0
    
    for inputs, targets in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    
    # Đánh giá trên tập Validation với k_list=[10, 20, 30]
    val_recalls = evaluate_recall_at_k(model, val_loader, k_list=[10, 20, 30])
    
    print(f"Epoch [{epoch:02d}/{num_epochs}] - Loss: {avg_loss:.4f} | "
          f"Val Recall@10: {val_recalls[10]:.4f} | "
          f"Val Recall@20: {val_recalls[20]:.4f} | "
          f"Val Recall@30: {val_recalls[30]:.4f}")
    
    # Lưu checkpoint tốt nhất dựa trên Recall@30
    if val_recalls[30] > best_val_recall30:
        best_val_recall30 = val_recalls[30]
        torch.save({
            "model_state_dict": model.state_dict(),
            "code2idx": code2idx,
            "idx2code": idx2code,
            "vocab_size": vocab_size
        }, best_model_path)
        print(f"  -->  Đã lưu mô hình tốt nhất vào: {best_model_path}")

print("\n HOÀN THÀNH HUẤN LUYỆN DOCTOR AI!")

 Bắt đầu quá trình huấn luyện mô hình Doctor AI...


Epoch [01/20] - Loss: 0.6582 | Val Recall@10: 0.0312 | Val Recall@20: 0.0312 | Val Recall@30: 0.0312
  -->  Đã lưu mô hình tốt nhất vào: ../models\doctor_ai_best.pt
Epoch [02/20] - Loss: 0.3772 | Val Recall@10: 0.0581 | Val Recall@20: 0.0581 | Val Recall@30: 0.0581
  -->  Đã lưu mô hình tốt nhất vào: ../models\doctor_ai_best.pt
Epoch [03/20] - Loss: 0.1484 | Val Recall@10: 0.0506 | Val Recall@20: 0.0506 | Val Recall@30: 0.0506
Epoch [04/20] - Loss: 0.1075 | Val Recall@10: 0.0596 | Val Recall@20: 0.0596 | Val Recall@30: 0.0596
  -->  Đã lưu mô hình tốt nhất vào: ../models\doctor_ai_best.pt
Epoch [05/20] - Loss: 0.0982 | Val Recall@10: 0.0596 | Val Recall@20: 0.0596 | Val Recall@30: 0.0596
Epoch [06/20] - Loss: 0.0945 | Val Recall@10: 0.0596 | Val Recall@20: 0.0596 | Val Recall@30: 0.0596
Epoch [07/20] - Loss: 0.0925 | Val Recall@10: 0.0596 | Val Recall@20: 0.0596 | Val Recall@30: 0.0596
Epoch [08/20] - Loss: 0.0903 | Val Recall@10: 0.0467 | Val Recall@20: 0.0467 | Val Recall@30: 0.0467


In [25]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_recalls = evaluate_recall_at_k(model, test_loader, k_list=[10, 20, 30])

print("KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (DOCTOR AI BLACK-BOX):")
print(f"  • Test Recall@10: {test_recalls[10]:.4f}")
print(f"  • Test Recall@20: {test_recalls[20]:.4f}")
print(f"  • Test Recall@30: {test_recalls[30]:.4f}")

KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (DOCTOR AI BLACK-BOX):
  • Test Recall@10: 0.0521
  • Test Recall@20: 0.0521
  • Test Recall@30: 0.0521
